# **Análisis de Movilidad - TfL London Stations**
Este proyecto analiza el flujo de pasajeros en la red de transporte de Londres en el período 2007-2021. A continuación, se detalla el diccionario de datos:

| Campo | Descripción |
| :--- | :---: |
| NLC | Código Nacional de Ubicación de la estación |
| Station | El nombre de la estación |
| En/Ex 20xx | El número total de entradas y salidas de una estación determinada en el año, desde 2007 hasta 2021 |
| LINES | Indica todas las líneas de TfL que operan en una estación determinada |
| NETWORK | Indica si la estación forma parte del metro de Londres |
| London Underground / Elizabeth Line / London Overground / DLR | Indica si esta estación forma parte de la red indicada |
| Night Tube? | Indica si una estación es parte del servicio nocturno de Tube y London Overground que opera los viernes y sábados por la noche |

# Parte 1

## Reconocimiento, Tratado y Limpieza de Nulls

In [88]:
import pandas as pd

tflStationData = pd.read_csv('TfL_stations.csv')

tflStationData.dropna(how='all', inplace=True)

porcentaje_de_nulls = tflStationData.isna().sum().sum()*100/tflStationData.size

print(porcentaje_de_nulls)

29.522935779816514


El cálculo anterior refiere al porcentaje de nulls que cuenta el archivo original y debemos tratar. Para ellos realizaremos un data cleaning y eventualmente, nuestro resultado, lo llevaremos a una Tercer Forma Normal (3FN) que nos permitirá el trabajo correcto de análisis.

Para poder convertir estos nulls en un dato consistente a su columna podremos tomar dos caminos que nos llevarán al mismo lugar, aunque realizando efectivamente el segundo para nuestro DataFrame.

Los enfoques son:
1. Tomar los tipos de datos de cada columna, para luego iterar sobre ellas en el DataFrame y, ante cada null, reemplazar por un valor consistente al dato correspondiente. Aquí desarrollo una lógica manual para entender la distribución de tipos de datos y aplicar una limpieza controlada.

In [89]:
#Primero obtenemos los tipos de datos de las columnas del DataFrame y los guardamos en una lista, si es que no fueron guardados previamente 
tipos_de_datos = []

for i in tflStationData.dtypes:
    tipo = i.name
    if (tipo not in tipos_de_datos):
        tipos_de_datos.append(tipo)


df_limpio_manual = tflStationData.copy() #Realizamos una copia para no modificar el DataFrame original y poder enseñar el ejemplo manual, ya que no se tomará este camino para el proyecto

#Ahora recorremos las columnas y aplicamos de forma manual, frente a un null, cada asignación 
for col in df_limpio_manual:

    if (df_limpio_manual[col].dtype.name in ['int64', 'float64']):
        df_limpio_manual[col] = df_limpio_manual[col].fillna(-1)
    else:
        df_limpio_manual[col] = df_limpio_manual[col].fillna('Sin texto')

porcentaje_de_nulls_manual = df_limpio_manual.isna().sum().sum()*100/df_limpio_manual.size

print(porcentaje_de_nulls_manual) #Output = 0.0, significa que se ha cumplido el objetivo ya que no hay nulls en el DataFrame df_limpio_manual

0.0


2. Uso de metodo nativos de Pandas (select_dtypes) para priorizar un código limpio, breve, legible y optimizado. 

In [90]:
lista_columnas_numericas = tflStationData.select_dtypes(include=['number']).columns

tflStationData[lista_columnas_numericas] = tflStationData[lista_columnas_numericas].fillna(-1)

lista_columnas_no_numericas = tflStationData.select_dtypes(exclude=['number']).columns

tflStationData[lista_columnas_no_numericas] = tflStationData[lista_columnas_no_numericas].fillna('Sin texto')

porcentaje_de_nulls_optimizado = tflStationData.isna().sum().sum()*100/tflStationData.size

print(porcentaje_de_nulls_optimizado) #Output = 0.0, completando la limpieza de nulls en nuestro DataFrame basandonos en metodos nativos y optimizados de Pandas

0.0


# Parte 2

## Reestructuración, Limpieza Final y Normalización

En esta parte la idea es la reestructuración del DataFrame para que se encuentre en una Tercer Forma Normal (3FN), y así poder realizar nuestras consultas y conclusiones en la Parte 3.

Al realizar una vista al archivo TFL_stations.csv notamos que hay una estructura ineficiente: si se desean agregar datos al paso de los años implica añadir columnas, cuando preferentemente deseamos añadir filas; columnas como "LINES" tiene múltiples valores separados por comas, quitando la atomicidad; datos que deberían ser ints pero resultan ser floats; y dos columnas (NETWORK y London Underground) que nos dan la misma información, siendo redundante.

En el siguiente código se encontrará la fundamentación de las aclaraciones previas y consultas de verificación (como por ejemplo, si hay filas duplicadas, entre otras consultas).

- 1) Secuencia de valores separados por comas en LINES. Para ello, la solución será atomizar cada conjunto de valores.

In [91]:
print(tflStationData['LINES'].str.contains(',').sum()) #Output = 102, corroborando que tenemos una lista de valores en 102 filas

tflStationData['LINES'] = tflStationData['LINES'].str.split(',').explode('LINES')

print(tflStationData['LINES'].str.contains(',').sum()) #Output = 0, ya no tenemos listas en LINES aunque sí filas por optimizar ya que multiplicamos su contenido al separar los valores de dicha columna

102
0


- 2) Filas duplicadas. Se corrobora la no existencia de filas duplicadas.

In [92]:
print(tflStationData.duplicated().sum()) #Output = 0, lo cual significa que hay 0 filas que son copias exactas de otra

0


- 3) Tipos de datos distintos en mismas columnas númericas (Int vs Float). La solución será unificar ello en columnas de ints (ya que será consistente con el dato de ingresos y egresos de personas).

In [93]:
print(tflStationData.dtypes) #El output enseñará que las columnas númericas estan dadas por float64

Unnamed: 0              int64
NLC                   float64
Station                object
En/Ex 2007            float64
En/Ex 2008            float64
En/Ex 2009            float64
En/Ex 2010            float64
En/Ex 2011            float64
En/Ex 2012            float64
En/Ex 2013            float64
En/Ex 2014            float64
En/Ex 2015            float64
En/Ex 2016            float64
En/Ex 2017            float64
En/Ex 2018            float64
En/Ex 2019            float64
En/Ex 2020            float64
En/Ex 2021            float64
LINES                  object
NETWORK                object
London Underground     object
Elizabeth Line         object
London Overground      object
DLR                    object
Night Tube?            object
dtype: object


In [94]:
columnas_mumericas = tflStationData.select_dtypes(include=['number']).columns.to_list()

tflStationData[columnas_mumericas] = tflStationData[columnas_mumericas].astype(int) #Casteamos al tipo de dato deseado, int

print(tflStationData.dtypes) #El output enseña que se ha hecho correctamente el casteo y trabajaremos con datos congruentes al contexto (números sin decimales)

Unnamed: 0             int64
NLC                    int64
Station               object
En/Ex 2007             int64
En/Ex 2008             int64
En/Ex 2009             int64
En/Ex 2010             int64
En/Ex 2011             int64
En/Ex 2012             int64
En/Ex 2013             int64
En/Ex 2014             int64
En/Ex 2015             int64
En/Ex 2016             int64
En/Ex 2017             int64
En/Ex 2018             int64
En/Ex 2019             int64
En/Ex 2020             int64
En/Ex 2021             int64
LINES                 object
NETWORK               object
London Underground    object
Elizabeth Line        object
London Overground     object
DLR                   object
Night Tube?           object
dtype: object


- 4) Columnas iguales, datos redundantes. La solución será, luego de verificar que realmente dan la misma información, la eliminación de una de ellas. Primero realizaremos la unificación de información, ya que ambas columnas serán iguales si sus filas reflejan la misma información, y lo hacen, pero no de la misma forma (Una dice "London Underground" y la otra "Yes"). Por eso, en cada lugar donde dice 'London Underground' en la columna NETWORK lo reemplazaremos por 'Yes', y luego realizaremos la consulta.

In [95]:
tflStationData['NETWORK'] = tflStationData['NETWORK'].replace('London Underground', 'Yes')

print(tflStationData['NETWORK'].equals(tflStationData['London Underground'])) #Output = True, por lo que procedemos con la eliminacion de la columna NETWORK

tflStationData = tflStationData.drop(columns=['NETWORK'])

True


Dado que ya realizamos las limpiezas necesarias pasaremos ahora a la reestructuración del DataFrame antes de conseguir que llegue a la 3FN. Para ello, el punto crítico más importante son las columnas que representan a los años y generan una ineficiencia en el guardado de datos. Entonces se añadirán dos columnas nuevas, "Año" y "Flujo_pasajeros" y se eliminarán todas aquellas columnas que representaban los ingresos y egresos por año (aunque manteniendo dicha información a modo de filas), para que añadir más datos implique un crecimiento de filas y no de columnas, dando una reorganización más útil y óptima.

In [96]:
df_a_modificar = tflStationData.filter(like='En/Ex') #Tomamos un nuevo DataFrame que contiene las columnas que mencionan los años

columnas_fijas = tflStationData.columns[~tflStationData.columns.str.contains('En/Ex')]

df_a_mantener = tflStationData[columnas_fijas] #Tomamos un nuevo DataFrame con las columnas que no contienen los años y quedarán como antes

df_final = tflStationData.melt(columnas_fijas.to_list(), df_a_modificar.columns.to_list(), 'Año', 'Flujo_Pasajeros') #Trabajaremos con el DataFrame df_final que ya contiene las columnas segun nuestra forma deseada

Dada nuestra nueva estructura, ahora retocaremos el DataFrame de la siguiente forma.

In [97]:

df_final['Año'] = df_final['Año'].str.replace(r'.*En/Ex\s*(.*)', r'\1', regex = True) #En la columna "Año" pasaremos de tener "En/Ex <año>" a solamente el año

df_final['Año'] = df_final['Año'].astype(int) #Casteamos la columna a int, ya que ahora solo tenemos números enteros y no strings

df_final = df_final.drop(columns=['Unnamed: 0']) #Quitamos la columna que no aportaba información al DataFrame

df_final[['London Underground', 'Elizabeth Line', 'London Overground', 'DLR']] = df_final[['London Underground', 'Elizabeth Line', 'London Overground', 'DLR']].replace('Sin texto', 'No') #Donde antes habían nulls y nuestra preocupación era quitarlos, ahora será ajustarlos al contenido de la columna, colocando un "No". La columna "Night Tube?" tiene "Yes" y "No", pero también nulls, por lo que ellos si quedarán como "Sin texto"

df_final.dtypes #Con esto corroboramos que se ha realizado el casteo y eliminado la columna correctamente

NLC                    int64
Station               object
LINES                 object
London Underground    object
Elizabeth Line        object
London Overground     object
DLR                   object
Night Tube?           object
Año                    int64
Flujo_Pasajeros        int64
dtype: object

Ordenamos por NLC ascendente y corroboramos su estado previo y siguiente

In [98]:
print(df_final['NLC'].equals(df_final['NLC'].sort_values(ascending= True))) #Output = False

df_final = df_final.sort_values(by = 'NLC', ascending = True)

print(df_final['NLC'].equals(df_final['NLC'].sort_values(ascending= True))) #Output = True

False
True


Resumidamente, se ha logrado una estructura que nos permitirá trabajar cómodamente luego. Para entender sobre donde estamos trabajando, debemos saber que la estructura actual es la siguiente:

df_final(NLC, Station, LINES, London Underground, Elizabeth Line, London Overground, DLR, Night Tube?, Año, Flujo_Pasajeros)

La idea actual es llegar a la 3FN, que para ello debemos primero ver que cumpla 1FN y 2FN:
- 1FN -> cumple ya que todas las columnas (inclusive LINES que fue tratada específicamente para corregir su estructura) tiene solamente valores atómicos, no hay grupos repetidos (fue eliminada la columna NETWORK que repetía información) y contamos con una clave primaria (en este caso, compuesta por NLC y Año).
- 2FN -> cumple 1FN, pero existen dependencias funcionales parciales; significa que hay atributos que dependen de parte de la clave (como Station con NLC) y no de su totalidad (del Año). Por lo que debemos dividir el DataFrame actual en 
- - Estaciones(NLC, Station, LINES, London Underground, Elizabeth Line, London Overground, DLR, Night Tube?)
- - Flujo(NLC, Año, Flujo_Pasajeros)

In [99]:
df_estaciones = df_final[['NLC', 'Station', 'LINES', 'London Underground', 'Elizabeth Line', 'London Overground', 'DLR', 'Night Tube?']]

df_estaciones = df_estaciones.rename(columns = {'Station' : 'Estacion', 'LINES' : 'Lineas', 'Night Tube?' : 'Metro_nocturno?'})

df_flujo = df_final[['NLC', 'Año', 'Flujo_Pasajeros']]

Es importante decir que el melt hecho para reestructurar el DataFrame ha multiplicado filas. Por lo que, antes de seguir analizando Formas Normales, se realizará la limpieza de duplicados.

In [100]:
print(df_estaciones.duplicated().sum())

df_estaciones = df_estaciones.drop_duplicates()

print(df_estaciones.duplicated().sum())

6104
0


- 3FN -> cumple 2FN y no hay dependencias transitivas entre atributos.

Por lo que nuestros DataFrames, df_estaciones y df_flujo, ya están listos para ser analizados en la Parte 3

# Parte 3

## Análisis de Datos

En esta parte el interés se encuentra en poder dar preguntas y respuestas que ya se encuentran en nuestros datos, pero hay que traducir para poder entender. Así es como se usarán las herramientas que la programación, SQL, Pandas y Python nos provee para darle valor al trabajo, asesorar y brindar sugerencias que concluyen del análisis.

Primero llevaremos nuestros DataFrames a archivos csv para trabajarlos en MySQL.

In [101]:
df_estaciones.to_csv('estaciones_limpias.csv', index = False, encoding = 'utf-8')

df_flujo.to_csv('flujo_pasajeros.csv', index = False, encoding = 'utf-8')

Dado esto, se realizarán los análisis de los años siguientes (junto a ellos se escribe el código trabajado para cada archivo y el motivo de selección del año):

- 2012, ya que Londres fue sede de los Juegos Olímpicos de dicho año.

In [102]:
df_olimpics = pd.read_csv('Weekly Data 2012.csv', skiprows=6) #Se saltea las primeras 6 líneas que no aportan información

df_olimpics.dropna(how='all', inplace = True)

df_olimpics.columns = df_olimpics.columns.str.strip().str.replace('\n', ' ').str.lower() #Con el .strip() eliminamos los vacios dentro de cada nombre de columna, dejando únicamente los strings que no son vacíos; con el .str.replace() reemplazamos los saltos de línea por espacios y con el .lower() hacemos que los nombres de las columnas queden en mínuscula, que nos es útil para luego trabajar bajo un criterio único

df_olimpics = df_olimpics.drop(columns=['note']) #Se decide eliminar esta columna ya que no aportaba información para las consultas

nuevas_columnas_2012 = {'nlc' : 'NLC', 
                   'station' : 'Estacion', 
                   'weekday' : 'Entradas_semana', 
                   'saturday' : 'Entradas_sabado', 
                   'sunday' : 'Entradas_domingo',
                   'weekday.1' : 'Salidas_semana', 
                   'saturday.1' : 'Salidas_sabado', 
                   'sunday.1' : 'Salidas_domingo', 
                   'million' : 'Entradas_salidas_anual_millones'}

df_olimpics = df_olimpics.rename(columns = nuevas_columnas_2012)

'''Se verifica que se cuenta con un DataFrame limpio'''

print(df_olimpics.isna().sum().sum()) #Output = 0, no hay nulls en el DataFrame

print(df_olimpics.duplicated().sum()) #Output = 0, no existen filas duplicadas

'''Se ordena por NLC descendente, se elimina la primer fila que está vacía y se reestructura el índice'''

print(df_olimpics['NLC'].equals(df_olimpics['NLC'].sort_values(ascending= True))) #Output = False

df_olimpics = df_olimpics.sort_values(by = 'NLC', ascending = True)

print(df_olimpics['NLC'].equals(df_olimpics['NLC'].sort_values(ascending= True))) #Output = True

df_olimpics = df_olimpics.drop(df_olimpics.index[0])

df_olimpics = df_olimpics.reset_index(drop = True)

df_olimpics.to_csv('movimientos_año_2012.csv', index = False, encoding = 'utf-8')

0
0
False
True


- 2017, ya que en el año 2016 se incorporó el servicio nocturno (Night Tube) y se desea comprender como afectó al movimiento viajes en las redes de metro una vez consolidado este sistema.

In [103]:
df_nocturno = pd.read_csv('Weekly Data 2017.csv', skiprows=6)

df_nocturno.dropna(how='all', inplace=True)

df_nocturno.columns = df_nocturno.columns.str.strip().str.replace('\n', ' ').str.lower()

df_nocturno = df_nocturno.drop(columns = ['note'])

nuevas_columnas = {'nlc' : 'NLC', 
                   'station' : 'Estacion',
                   'borough' : 'Ciudad', 
                   'weekday' : 'Entradas_semana', 
                   'saturday' : 'Entradas_sabado', 
                   'sunday' : 'Entradas_domingo', 
                   'weekday.1' : 'Salidas_semana', 
                   'saturday.1' : 'Salidas_sabado', 
                   'sunday.1' : 'Salidas_domingo', 
                   'million' : 'Entradas_salidas_anual_millones'}

df_nocturno = df_nocturno.rename(columns = nuevas_columnas)

df_nocturno = df_nocturno.drop(columns = ['Ciudad'])

'''Se verifica que se cuenta con un DataFrame limpio'''

print(df_nocturno.isna().sum().sum()) #Output = 0, no hay nulls en el DataFrame

print(df_nocturno.duplicated().sum()) #Output = 0, no existen filas duplicadas

'''Se ordena por NLC descendente y se reestructura el índice'''

print(df_nocturno['NLC'].equals(df_nocturno['NLC'].sort_values(ascending= True))) #Output = False

df_nocturno = df_nocturno.sort_values(by = 'NLC', ascending = True)

print(df_nocturno['NLC'].equals(df_nocturno['NLC'].sort_values(ascending= True))) #Output = True

df_nocturno = df_nocturno.reset_index(drop=True)

df_nocturno.to_csv('movimientos_año_2017.csv', index = False, encoding = 'utf-8')

0
0
False
True


- 2021, un año pasada la pandemia del 2020 que permite entender como se transformó la ciudad y adapto la población su movimiento junto a sus nuevas rutinas.

In [104]:
df_pandemia = pd.read_csv('AC2021_AnnualisedEntryExit - Annualised.csv', skiprows = 6)

df_pandemia.dropna(how = 'all', inplace = True)

df_pandemia = df_pandemia.drop(columns = ['Mode', 'ASC', 'Coverage', 'Source'])

df_pandemia['Entradas_semana'] = df_pandemia['Entries'] + df_pandemia['Entries.1']

df_pandemia['Salidas_semana'] = df_pandemia['Exits'] + df_pandemia['Exits.1']

nuevas_columnas_2017 = {'Entries.2' : 'Entradas_sabado',
                        'Entries.3' : 'Entradas_domingo',
                        'Exits.2' : 'Salidas_sabado',
                        'Exits.3' : 'Salidas_domingo',
                        'En/Ex' : 'Entradas_salidas_anual_millones',
                        'Station' : 'Estacion'}

df_pandemia = df_pandemia.rename(columns = nuevas_columnas_2017)

df_pandemia = df_pandemia.drop(columns = ['Entries', 'Entries.1', 'Exits', 'Exits.1'])

df_pandemia = df_pandemia[['NLC', 'Estacion', 'Entradas_semana', 'Entradas_sabado', 'Entradas_domingo', 'Salidas_semana', 'Salidas_sabado', 'Salidas_domingo', 'Entradas_salidas_anual_millones']]

'''Se verifica que se cuenta con un DataFrame limpio'''

print(df_pandemia.isna().sum().sum()) #Output = 0, no hay nulls en el DataFrame

print(df_pandemia.duplicated().sum()) #Output = 0, no existen filas duplicadas

'''Se ordena por NLC descendente, se elimina la primer fila que está vacía y se reestructura el índice'''

print(df_pandemia['NLC'].equals(df_pandemia['NLC'].sort_values(ascending= True))) #Output = False

df_pandemia = df_pandemia.sort_values(by = 'NLC', ascending = True)

print(df_pandemia['NLC'].equals(df_pandemia['NLC'].sort_values(ascending= True))) #Output = True

df_pandemia = df_pandemia.reset_index(drop = True)

df_pandemia.to_csv('movimientos_año_2021.csv', index = False, encoding = 'utf-8')

0
0
False
True


# Parte 4

## Visualización de Datos